# Supervised Fine-Tuning for the NVIDIA Nemotron Model Reasoning Challenge

This notebook implements stage 1 of the pipeline: supervised fine-tuning (SFT) for `nvidia/Nemotron-3-Nano-30B` using Hugging Face, PyTorch, and PEFT/LoRA. The goal is to improve reasoning accuracy on the challenge benchmark while keeping the output format compatible with the competition evaluator and with a later GRPO/PPO stage.

Key decisions in this notebook:
- Train only a LoRA adapter with rank `<= 32`, matching the submission constraint.
- Keep the final answer inside `\boxed{}` so the local validation setup mirrors the competition evaluator.
- Use an RL-compatible response schema now, supervising `{thought}` directly from the `generated_cot` column in `outputs/SFT_cot_bit_manipulation_data.csv`.
- Save a ready-to-zip adapter folder that contains `adapter_config.json` for submission packaging.


## 1. Install the training stack

This notebook stays inside the Hugging Face + PyTorch ecosystem. `bitsandbytes` is used for 4-bit loading so the 30B base model can be adapted with QLoRA-style training, and `peft` is used to create the rank-32 LoRA adapter required by the competition.


In [1]:
%pip install -q -U transformers peft accelerate bitsandbytes datasets pandas scikit-learn sentencepiece safetensors


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 38.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 100.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 75.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 14.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.2 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.2 which is incompatible.
bqplot 0.12.45 requires pandas<3.0.0,>=1.0.0, but you have pandas 3.0.2 which is incompa

## 2. Import libraries and define the SFT configuration

The configuration below is designed around the competition rules and your planned two-stage workflow:
- `BASE_MODEL_ID` points to the Nemotron 3 Nano 30B base model.
- LoRA rank is capped at `32`.
- The response template already uses `Thought` and `Solution` sections so later RL can keep the same interface.
- The SFT dataset already includes `prompt`, `answer`, and `generated_cot`, so the notebook supervises the full `Thought` and `Solution` format directly.


In [2]:
import gc
import inspect
import json
import math
import os
import random
import re
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from IPython.display import display
from sklearn.model_selection import train_test_split
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    DataCollatorForSeq2Seq,
    StoppingCriteria,
    StoppingCriteriaList,
    Trainer,
    TrainingArguments,
    set_seed,
)
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

BASE_MODEL_ID = "HuggingFaceTB/SmolLM2-135M" #"nvidia/Nemotron-3-Nano-30B"
TRAIN_CSV = Path("outputs/SFT_cot_bit_manipulation_data.csv")
TEST_CSV = Path("test.csv")
OUTPUT_DIR = Path("outputs/nemotron_sft_stage1")
ADAPTER_DIR = OUTPUT_DIR / "final_adapter"
SUBMISSION_ZIP = Path("submission.zip")
VALIDATION_REPORT_TXT = OUTPUT_DIR / "local_validation_preview.txt"

MAX_SEQ_LENGTH = 4096
MAX_NEW_TOKENS =
VALID_SIZE = 0.02
VALIDATION_SAMPLE_SIZE = 8
SOLUTION_END_MARKER = "<|end_of_solution|>"
LOCAL_NUMERIC_REL_TOL = 1e-4  # Local proxy only. Adjust if the official metric uses a different tolerance.

LORA_RANK = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0.05
LEARNING_RATE = 2e-4
NUM_EPOCHS = 1
TRAIN_BATCH_SIZE = 1
EVAL_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 16
WARMUP_RATIO = 0.03
WEIGHT_DECAY = 0.01
LOGGING_STEPS = 10
EVAL_STEPS = 100
SAVE_STEPS = 100

assert LORA_RANK <= 32, "Competition submissions require LoRA rank <= 32."

SYSTEM_INSTRUCTION = """Your role as an assistant involves thoroughly exploring questions through a systematic long thinking process before providing the final precise and accurate solution. Structure every response into two sections named Thought and Solution. In the Thought section, reason inside <|begin_of_thought|> and <|end_of_thought|>. In the Solution section, present the final answer inside <|begin_of_solution|> and <|end_of_solution|>. Always place the final answer inside \\boxed{}."""

PROMPT_TEMPLATE = """System:
{system_instruction}

User:
{problem}

Assistant:
"""

RESPONSE_TEMPLATE = """Thought:
<|begin_of_thought|>
{thought}
<|end_of_thought|>

Solution:
<|begin_of_solution|>
The final answer is \\boxed{{{answer}}}.
<|end_of_solution|>
"""

def get_compute_dtype():
    if not torch.cuda.is_available():
        return torch.float32
    major, _ = torch.cuda.get_device_capability(0)
    return torch.bfloat16 if major >= 8 else torch.float16

COMPUTE_DTYPE = get_compute_dtype()
USE_4BIT = torch.cuda.is_available()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print({
    "base_model_id": BASE_MODEL_ID,
    "compute_dtype": str(COMPUTE_DTYPE),
    "use_4bit": USE_4BIT,
    "max_seq_length": MAX_SEQ_LENGTH,
    "lora_rank": LORA_RANK,
})


{'base_model_id': 'HuggingFaceTB/SmolLM2-135M', 'compute_dtype': 'torch.float16', 'use_4bit': True, 'max_seq_length': 4096, 'lora_rank': 32}


## 3. Load the competition dataset

This notebook now trains from `outputs/SFT_cot_bit_manipulation_data.csv`, which includes `prompt`, `answer`, and `generated_cot`. That lets SFT supervise the `Thought` section from the dataset itself instead of relying on a generic scaffold.


In [4]:
train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)

required_train_columns = {"id", "prompt", "answer", "generated_cot"}
required_test_columns = {"id", "prompt"}

assert required_train_columns.issubset(train_df.columns), train_df.columns.tolist()
assert required_test_columns.issubset(test_df.columns), test_df.columns.tolist()

print("train rows:", len(train_df))
print("test rows:", len(test_df))
display(train_df.head(3))


train rows: 298
test rows: 3


,id,prompt,answer,generated_cot,type,confidence,method,n_ambig_bits,generated_answer
0,00754598,"In Alice's Wonderland, a secret bit manipulati...",11101111,I need to find the secret bit manipulation rul...,Bit Manipulation,high,ctx,0,11101111
1,00fdc0be,"In Alice's Wonderland, a secret bit manipulati...",11111111,I need to find the secret bit manipulation rul...,Bit Manipulation,high,w_mix,0,11111111
2,01248b76,"In Alice's Wonderland, a secret bit manipulati...",11000101,I need to find the secret bit manipulation rul...,Bit Manipulation,high,w_mix,0,11000101


## 4. Build an RL-compatible SFT target format

This is the main design change from a plain `\boxed{answer}` target. The prompt already uses the same high-level interface you want later for GRPO/PPO, and the supervised target now uses the rationale data already present in your SFT CSV:
- The `Thought` section is populated from each row's `generated_cot` value.
- The `Solution` section always contains the boxed final answer expected by the evaluator.
- Later RL can keep the same format and learn richer reasoning trajectories without changing the outer conversation contract.


In [5]:
def build_prompt(problem: str) -> str:
    return PROMPT_TEMPLATE.format(
        system_instruction=SYSTEM_INSTRUCTION.strip(),
        problem=str(problem).strip(),
    )

def build_target(answer: str, thought: str) -> str:
    return RESPONSE_TEMPLATE.format(
        thought=thought.strip(),
        answer=str(answer).strip(),
    )

example_prompt = build_prompt(train_df.loc[0, "prompt"])
example_target = build_target(
    train_df.loc[0, "answer"],
    train_df.loc[0, "generated_cot"],
)

print(example_prompt[:1200])
print("\n--- TARGET PREVIEW ---\n")
print(example_target)


System:
Your role as an assistant involves thoroughly exploring questions through a systematic long thinking process before providing the final precise and accurate solution. Structure every response into two sections named Thought and Solution. In the Thought section, reason inside <|begin_of_thought|> and <|end_of_thought|>. In the Solution section, present the final answer inside <|begin_of_solution|> and <|end_of_solution|>. Always place the final answer inside \boxed{}.

User:
In Alice's Wonderland, a secret bit manipulation rule transforms 8-bit binary numbers. The transformation involves operations like bit shifts, rotations, XOR, AND, OR, NOT, and possibly majority or choice functions.

Here are some examples of input -> output:
10101001 -> 01100010
00110011 -> 10010100
00100111 -> 00110001
11011100 -> 11010111
01110010 -> 10001100
01010001 -> 10011100
00100101 -> 00100001
00100001 -> 00000000
10011111 -> 11011111

Now, determine the output for: 01111110

Assistant:


--- TARGE

## 5. Split the data and render the training texts

A small validation split is useful for local checks before packaging the adapter. Each row becomes a prompt-completion pair where the model sees the full conversation prefix and learns only the assistant completion.


In [6]:
train_split, valid_split = train_test_split(
    train_df,
    test_size=VALID_SIZE,
    random_state=SEED,
    shuffle=True,
)

train_split = train_split.reset_index(drop=True).copy()
valid_split = valid_split.reset_index(drop=True).copy()

for frame in (train_split, valid_split):
    frame["prompt_text"] = frame["prompt"].apply(build_prompt)
    frame["target_text"] = frame.apply(
        lambda row: build_target(row["answer"], row["generated_cot"]),
        axis=1,
    )

print("train rows:", len(train_split))
print("valid rows:", len(valid_split))
display(train_split[["id", "prompt_text", "target_text"]].head(1))


train rows: 292
valid rows: 6


,id,prompt_text,target_text
0,53329505,System:\nYour role as an assistant involves th...,Thought:\n<|begin_of_thought|>\nI need to find...


## 6. Load the tokenizer and the 4-bit base model for QLoRA

The challenge requires the final submission to be a LoRA adapter for the Nemotron 30B base model. This cell loads that base model and prepares it for parameter-efficient fine-tuning. The helper for `target_modules` is intentionally defensive so the notebook still works if the model uses slightly different internal projection names.

Resource note: a 30B model is much larger than a typical Kaggle toy setup. You should expect to run this on a high-memory GPU or a multi-GPU environment.


In [7]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=quant_config,
    dtype=COMPUTE_DTYPE,
    device_map="auto",
    trust_remote_code=True,
    low_cpu_mem_usage=True,
)
model = prepare_model_for_kbit_training(model)

def find_lora_target_modules(model):
    preferred = {"q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"}
    hits = sorted({name.split(".")[-1] for name, _ in model.named_modules() if name.split(".")[-1] in preferred})
    if hits:
        return hits

    fallback = set()
    linear_like = {"Linear", "Linear4bit", "Linear8bitLt"}
    for name, module in model.named_modules():
        leaf_name = name.split(".")[-1]
        if leaf_name == "lm_head":
            continue
        if module.__class__.__name__ in linear_like:
            fallback.add(leaf_name)
    return sorted(fallback)

target_modules = find_lora_target_modules(model)
print("LoRA target modules:", target_modules)

lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=target_modules,
)

model = get_peft_model(model, lora_config)
model.gradient_checkpointing_enable()
model.enable_input_require_grads()
model.config.use_cache = False
model.print_trainable_parameters()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/704 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/269M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

LoRA target modules: ['down_proj', 'gate_proj', 'k_proj', 'o_proj', 'q_proj', 'up_proj', 'v_proj']
trainable params: 9,768,960 || all params: 144,283,968 || trainable%: 6.7706


## 7. Tokenize with answer-only loss masking

The model should learn to generate only the assistant completion, not to copy the prompt. To enforce that, the prompt tokens are masked with `-100` and loss is applied only to the target completion. This is the standard setup for causal language model SFT when prompts and completions are concatenated.


In [8]:
train_dataset = Dataset.from_pandas(train_split[["id", "prompt_text", "target_text"]], preserve_index=False)
valid_dataset = Dataset.from_pandas(valid_split[["id", "prompt_text", "target_text"]], preserve_index=False)

def tokenize_example(example):
    prompt_text = example["prompt_text"]
    target_text = example["target_text"] + tokenizer.eos_token
    full_text = prompt_text + target_text

    prompt_tokens = tokenizer(
        prompt_text,
        add_special_tokens=False,
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
    )
    full_tokens = tokenizer(
        full_text,
        add_special_tokens=False,
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
    )

    input_ids = full_tokens["input_ids"]
    attention_mask = full_tokens["attention_mask"]
    prompt_len = min(len(prompt_tokens["input_ids"]), max(len(input_ids) - 1, 0))
    labels = [-100] * prompt_len + input_ids[prompt_len:]

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }

tokenized_train = train_dataset.map(tokenize_example, remove_columns=train_dataset.column_names)
tokenized_valid = valid_dataset.map(tokenize_example, remove_columns=valid_dataset.column_names)

tokenized_train = tokenized_train.filter(lambda row: any(label != -100 for label in row["labels"]))
tokenized_valid = tokenized_valid.filter(lambda row: any(label != -100 for label in row["labels"]))

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    pad_to_multiple_of=8,
    label_pad_token_id=-100,
)

print(tokenized_train[0].keys())
print("tokenized train rows:", len(tokenized_train))
print("tokenized valid rows:", len(tokenized_valid))


Map:   0%|          | 0/292 [00:00<?, ? examples/s]

Map:   0%|          | 0/6 [00:00<?, ? examples/s]

Filter:   0%|          | 0/292 [00:00<?, ? examples/s]

Filter:   0%|          | 0/6 [00:00<?, ? examples/s]

dict_keys(['input_ids', 'attention_mask', 'labels'])
tokenized train rows: 292
tokenized valid rows: 6


## 8. Configure the Hugging Face trainer

The training arguments below are a practical starting point for stage-1 SFT. They favor stable QLoRA training and keep evaluation periodic so you can monitor loss before moving into RL. If your hardware budget is larger, you can scale sequence length, epochs, and validation cadence later.


In [9]:
training_args_kwargs = dict(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    logging_steps=LOGGING_STEPS,
    logging_first_step=True,
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    bf16=(COMPUTE_DTYPE == torch.bfloat16),
    fp16=(COMPUTE_DTYPE == torch.float16),
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    max_grad_norm=0.3,
    report_to="none",
    remove_unused_columns=False,
    gradient_checkpointing=True,
    dataloader_num_workers=2,
    seed=SEED,
)

training_args_signature = inspect.signature(TrainingArguments.__init__)
if "eval_strategy" in training_args_signature.parameters:
    training_args_kwargs["eval_strategy"] = "steps"
elif "evaluation_strategy" in training_args_signature.parameters:
    training_args_kwargs["evaluation_strategy"] = "steps"

if "group_by_length" in training_args_signature.parameters:
    training_args_kwargs["group_by_length"] = True

training_args = TrainingArguments(**training_args_kwargs)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_valid,
    data_collator=data_collator,
)

trainer


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


## 9. Run supervised fine-tuning and save the adapter

This cell performs stage-1 SFT and then saves the LoRA adapter plus tokenizer files into `outputs/nemotron_sft_stage1/final_adapter`. That directory is the one we will later zip into `submission.zip`.


In [10]:
train_result = trainer.train()
trainer.save_model(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))

metrics = train_result.metrics
with open(OUTPUT_DIR / "train_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

metrics


Step,Training Loss,Validation Loss
19,0.981546,0.664928


{'train_runtime': 246.6746,
 'train_samples_per_second': 1.184,
 'train_steps_per_second': 0.077,
 'total_flos': 200333534736384.0,
 'train_loss': 0.8990647981041356,
 'epoch': 1.0}

In [14]:
train_result = trainer.train()
trainer.save_model(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))

metrics = train_result.metrics
with open(OUTPUT_DIR / "train_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

metrics

Step,Training Loss,Validation Loss
19,0.586506,0.280339


{'train_runtime': 209.6223,
 'train_samples_per_second': 1.393,
 'train_steps_per_second': 0.091,
 'total_flos': 200333534736384.0,
 'train_loss': 0.5162704618353593,
 'epoch': 1.0}

## 10. Reload the trained adapter and run a local validation check

This validation block is intentionally independent from the training step. It reloads the saved adapter from disk, attaches it to the base model, runs deterministic local inference, and writes the full preview table to a text file for later inspection.

The official leaderboard will score your adapter in `vLLM`, but a local proxy is still useful. This section follows the competition spirit by:
- generating deterministically with `temperature=0.0` and `top_p=1.0`,
- extracting the last `\boxed{}` answer when present,
- falling back to numeric parsing if needed, and
- reporting a small validation accuracy estimate before packaging the adapter.


In [15]:
MAX_NEW_TOKENS = 4096
boxed_pattern = re.compile(r"\\boxed\{([^}]*)\}")
number_pattern = re.compile(r"-?\d+(?:\.\d+)?")
preview_columns = ["id", "target", "prediction", "match", "generated_text"]

class StopOnSubsequence(StoppingCriteria):
    def __init__(self, stop_token_ids):
        self.stop_token_ids = stop_token_ids

    def __call__(self, input_ids, scores, **kwargs):
        if not self.stop_token_ids:
            return False
        if input_ids.shape[1] < len(self.stop_token_ids):
            return False
        return input_ids[0, -len(self.stop_token_ids):].tolist() == self.stop_token_ids

def extract_prediction(text: str) -> str:
    boxed_matches = boxed_pattern.findall(text)
    if boxed_matches:
        return boxed_matches[-1].strip()

    numeric_matches = number_pattern.findall(text)
    if numeric_matches:
        return numeric_matches[-1].strip()

    lines = [line.strip() for line in text.splitlines() if line.strip()]
    return lines[-1] if lines else text.strip()

def normalize_answer(text: str) -> str:
    return str(text).strip()

def try_float(text: str):
    cleaned = normalize_answer(text).replace(",", "")
    try:
        return float(cleaned)
    except ValueError:
        return None

def competition_style_match(prediction: str, target: str, rel_tol: float = LOCAL_NUMERIC_REL_TOL) -> bool:
    pred = normalize_answer(prediction)
    truth = normalize_answer(target)
    if pred == truth:
        return True

    pred_num = try_float(pred)
    truth_num = try_float(truth)
    if pred_num is not None and truth_num is not None:
        return math.isclose(pred_num, truth_num, rel_tol=rel_tol, abs_tol=rel_tol)

    return False

def load_inference_artifacts():
    assert ADAPTER_DIR.exists(), f"Missing adapter directory: {ADAPTER_DIR}"

    # if "trainer" in globals():
    #     del trainer
    # if "model" in globals():
    #     del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    inference_tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR, trust_remote_code=True)
    if inference_tokenizer.pad_token is None:
        inference_tokenizer.pad_token = inference_tokenizer.eos_token
    inference_tokenizer.padding_side = "right"

    quant_config = None
    model_kwargs = {
        "trust_remote_code": True,
        "low_cpu_mem_usage": True,
    }
    if torch.cuda.is_available():
        quant_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=COMPUTE_DTYPE,
        )
        model_kwargs["device_map"] = "auto"
        model_kwargs["quantization_config"] = quant_config
        model_kwargs["torch_dtype"] = COMPUTE_DTYPE
    else:
        model_kwargs["torch_dtype"] = torch.float32

    base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL_ID, **model_kwargs)
    inference_model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
    inference_model.eval()

    stop_token_ids = inference_tokenizer.encode(SOLUTION_END_MARKER, add_special_tokens=False)
    stopping_criteria = StoppingCriteriaList([StopOnSubsequence(stop_token_ids)]) if stop_token_ids else None
    return inference_model, inference_tokenizer, stopping_criteria

def generate_answer(problem: str, inference_model, inference_tokenizer, stopping_criteria, max_new_tokens: int = MAX_NEW_TOKENS) -> str:
    prompt_text = build_prompt(problem)
    device = next(inference_model.parameters()).device
    inputs = inference_tokenizer(
        prompt_text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
    ).to(device)

    if hasattr(inference_model, "gradient_checkpointing_disable"):
        inference_model.gradient_checkpointing_disable()

    previous_use_cache = getattr(inference_model.config, "use_cache", True)
    inference_model.config.use_cache = True
    inference_model.eval()

    try:
        with torch.no_grad():
            outputs = inference_model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                use_cache=True,
                eos_token_id=inference_tokenizer.eos_token_id,
                pad_token_id=inference_tokenizer.pad_token_id,
                stopping_criteria=stopping_criteria,
            )
    finally:
        inference_model.config.use_cache = previous_use_cache

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    generated_text = inference_tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

    if SOLUTION_END_MARKER in generated_text:
        generated_text = generated_text.split(SOLUTION_END_MARKER, 1)[0] + SOLUTION_END_MARKER

    return generated_text

inference_model, inference_tokenizer, stopping_criteria = load_inference_artifacts()

sampled_valid = valid_split.sample(
    n=min(VALIDATION_SAMPLE_SIZE, len(valid_split)),
    random_state=SEED,
).reset_index(drop=True)
rows = []

print({"validation_sample_size": len(sampled_valid), "max_new_tokens": MAX_NEW_TOKENS, "report_path": str(VALIDATION_REPORT_TXT)})
for row_idx, row in enumerate(sampled_valid.itertuples(index=False), start=1):
    generated = generate_answer(row.prompt, inference_model, inference_tokenizer, stopping_criteria)
    prediction = extract_prediction(generated)
    target = str(row.answer).strip()
    match = competition_style_match(prediction, target)

    print(f"[{row_idx}/{len(sampled_valid)}] id={row.id} target={target} prediction={prediction} match={match}")
    rows.append({
        "id": row.id,
        "target": target,
        "prediction": prediction,
        "match": match,
        "generated_text": generated,
    })

preview_df = pd.DataFrame(rows, columns=preview_columns)
local_proxy_accuracy = preview_df["match"].mean() if len(preview_df) else float("nan")
print({"local_proxy_accuracy": local_proxy_accuracy, "sample_size": len(preview_df)})
display(preview_df[["id", "target", "prediction", "match"]])
display(preview_df[["id", "generated_text"]].head(3))

preview_df[preview_columns].to_string(VALIDATION_REPORT_TXT, index=False)
print(f"Saved validation preview to {VALIDATION_REPORT_TXT.resolve()}")


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

{'validation_sample_size': 6, 'max_new_tokens': 160, 'report_path': 'outputs/nemotron_sft_stage1/local_validation_preview.txt'}
[1/6] id=8abcff0f target=11110111 prediction=0101 match=False
[2/6] id=e568d07b target=11110001 prediction=0000 match=False
[3/6] id=c628dd06 target=10010110 prediction=0010 match=False
[4/6] id=d8ef1dae target=11011011 prediction=0100 match=False
[5/6] id=410a5cbd target=11111100 prediction=0111 match=False
[6/6] id=084a4496 target=11000111 prediction=1100 match=False
{'local_proxy_accuracy': np.float64(0.0), 'sample_size': 6}


,id,target,prediction,match
0,8abcff0f,11110111,0101,False
1,e568d07b,11110001,0000,False
2,c628dd06,10010110,0010,False
3,d8ef1dae,11011011,0100,False
4,410a5cbd,11111100,0111,False
5,084a4496,11000111,1100,False


,id,generated_text
0,8abcff0f,Thought:\n<|begin_of_thought|>\nI need to find...
1,e568d07b,Thought:\n<|begin_of_thought|>\nI need to find...
2,c628dd06,Thought:\n<|begin_of_thought|>\nI need to find...


Saved validation preview to /content/outputs/nemotron_sft_stage1/local_validation_preview.txt


## 11. Package the adapter as `submission.zip`

The competition expects a zip file containing a compatible LoRA adapter and an `adapter_config.json`. This cell verifies that the saved rank respects the challenge constraint and then creates the final archive.


In [ ]:
adapter_config_path = ADAPTER_DIR / "adapter_config.json"
assert adapter_config_path.exists(), f"Missing {adapter_config_path}"

with open(adapter_config_path) as f:
    adapter_config = json.load(f)

assert adapter_config.get("r") <= 32, adapter_config

with zipfile.ZipFile(SUBMISSION_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for file_path in ADAPTER_DIR.rglob("*"):
        if file_path.is_file():
            zf.write(file_path, arcname=file_path.relative_to(ADAPTER_DIR))

print("Created:", SUBMISSION_ZIP.resolve())
print("Archive size (MB):", round(SUBMISSION_ZIP.stat().st_size / (1024 ** 2), 2))
print("Adapter rank:", adapter_config.get("r"))
print("Archive contents:")
with zipfile.ZipFile(SUBMISSION_ZIP, "r") as zf:
    for name in zf.namelist():
        print(" -", name)


## 12. Why this SFT layout is the right hand-off into RL

Your current SFT CSV already contains a `generated_cot` column, so this notebook can directly supervise both the `Thought` and `Solution` sections before GRPO/PPO. That means the model is already learning three things that are useful for the RL hand-off:
- the conversation contract you want to keep later (`Thought` + `Solution`),
- the requirement that the final answer lands in `\boxed{}`, and
- the task distribution and answer space of the competition.

If you later want to refine the `Thought` quality even further, the clean extension is to improve or regenerate the `generated_cot` teacher traces and run another distillation pass before RL.
